# Tool 2: fetch_multispectral_imagery (Sentinel-2 L2A)

This tool retrieves **calibrated Surface Reflectance multi-band rasters (FLOAT32)** for arbitrary spectral bands (B01-B12, SCL, AOT, etc.) in GeoTIFF format with **AOI-Level SCL Quality Validation**.

In [ ]:
# 1. Install Dependencies
!pip -q install requests rasterio numpy pydantic fastmcp

In [ ]:
# 3. Configuration
# ==============================================================================
# API CREDENTIAL CONFIGURATION:
# This tool requires Sentinel Hub OAuth credentials (CLIENT_ID and CLIENT_SECRET).
# Generate your credentials from the official Sentinel Hub Dashboard:
# https://apps.sentinel-hub.com/dashboard/#/account/settings
#
# Supply them via environment variables:
#   export SENTINEL_CLIENT_ID="your_client_id_here"
#   export SENTINEL_CLIENT_SECRET="your_client_secret_here"
# (Alternative aliases: SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET)
# or in Google Colab userdata: userdata.get('SENTINEL_CLIENT_ID')
# ==============================================================================
try:
    from google.colab import userdata
    CLIENT_ID = userdata.get("SENTINEL_CLIENT_ID") or os.getenv("SENTINEL_CLIENT_ID") or os.getenv("SENTINELHUB_CLIENT_ID")
    CLIENT_SECRET = userdata.get("SENTINEL_CLIENT_SECRET") or os.getenv("SENTINEL_CLIENT_SECRET") or os.getenv("SENTINELHUB_CLIENT_SECRET")
except ImportError:
    CLIENT_ID = os.getenv("SENTINEL_CLIENT_ID") or os.getenv("SENTINELHUB_CLIENT_ID")
    CLIENT_SECRET = os.getenv("SENTINEL_CLIENT_SECRET") or os.getenv("SENTINELHUB_CLIENT_SECRET")

TOKEN_URL = "https://services.sentinel-hub.com/auth/realms/main/protocol/openid-connect/token"
PROCESS_URL = "https://services.sentinel-hub.com/process/v1"
CATALOG_URL = "https://services.sentinel-hub.com/api/v1/catalog/1.0.0/search"

@dataclass(frozen=True)
class ToolConfig:
    client_id: Optional[str]
    client_secret: Optional[str]
    token_url: str = TOKEN_URL
    process_url: str = PROCESS_URL
    catalog_url: str = CATALOG_URL

CONFIG = ToolConfig(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)

if not CLIENT_ID or not CLIENT_SECRET:
    print("NOTICE: Sentinel Hub credentials (SENTINEL_CLIENT_ID / SENTINEL_CLIENT_SECRET) are not set.")
    print("Please set environment variables before executing API requests. See README.md for instructions.")
else:
    print("Tool 2 Configuration initialized with environment credentials.")


In [ ]:
# 4. Resilient OAuth Token Manager
class TokenManager:
    def __init__(self, config: ToolConfig, http_client: "ResilientHTTPClient"):
        self.config = config
        self.http_client = http_client
        self._access_token = None
        self._expires_at = 0

    def get_token(self) -> str:
        current_time = time.time()
        if self._access_token is not None and current_time < self._expires_at:
            return self._access_token

        if not self.config.client_id or not self.config.client_secret:
            raise RuntimeError(
                "Sentinel Hub credentials missing! Please configure 'SENTINEL_CLIENT_ID' and "
                "'SENTINEL_CLIENT_SECRET' environment variables before running satellite queries. "
                "Generate credentials at https://apps.sentinel-hub.com/dashboard/#/account/settings"
            )

        response = self.http_client.post(
            self.config.token_url,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={
                "grant_type": "client_credentials",
                "client_id": self.config.client_id,
                "client_secret": self.config.client_secret,
            },
        )
        if not response.ok:
            raise RuntimeError(f"Failed to acquire Sentinel Hub OAuth token: {response.status_code} - {response.text}")

        token_data = response.json()
        access_token = token_data.get("access_token")
        expires_in = token_data.get("expires_in", 3600)
        if not access_token or not expires_in:
            raise RuntimeError("Invalid token response from Sentinel Hub auth server.")

        self._access_token = access_token
        self._expires_at = current_time + expires_in - 60
        return self._access_token


In [ ]:
# 4. Authentication Manager (Token Caching)
class SentinelHubAuth:
    def __init__(self, config: ToolConfig):
        self.config = config
        self._access_token = None
        self._expires_at = 0.0

    def get_token(self) -> str:
        current_time = time.time()
        if self._access_token is not None and current_time < self._expires_at:
            return self._access_token

        response = requests.post(
            self.config.token_url,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={
                "grant_type": "client_credentials",
                "client_id": self.config.client_id,
                "client_secret": self.config.client_secret,
            },
            timeout=30,
        )
        if not response.ok:
            try: error_data = response.json()
            except ValueError: error_data = response.text
            raise RuntimeError(f"Sentinel Hub auth failed (HTTP {response.status_code}): {error_data}")

        token_data = response.json()
        access_token = token_data.get("access_token")
        expires_in = token_data.get("expires_in")
        if not access_token or not expires_in:
            raise RuntimeError("Invalid token response from Sentinel Hub.")

        self._access_token = access_token
        self._expires_at = current_time + float(expires_in) - 60
        return self._access_token

auth = SentinelHubAuth(CONFIG)

In [ ]:
# 5. Resilient HTTP Client
class SentinelHubHTTPClient:
    def __init__(self, auth: SentinelHubAuth, max_retries: int = 3, timeout: int = 60):
        self.auth = auth
        self.max_retries = max_retries
        self.timeout = timeout
        self.retryable_status_codes = {429, 500, 502, 503, 504}

    def _get_headers(self, content_type: str = "application/json") -> dict:
        token = self.auth.get_token()
        return {"Authorization": f"Bearer {token}", "Content-Type": content_type}

    def request(self, method: str, url: str, *, content_type: str = "application/json", **kwargs) -> requests.Response:
        for attempt in range(self.max_retries + 1):
            try:
                response = requests.request(
                    method=method,
                    url=url,
                    headers=self._get_headers(content_type),
                    timeout=self.timeout,
                    **kwargs,
                )
            except requests.RequestException:
                if attempt >= self.max_retries: raise
                time.sleep(min(2 ** attempt, 30))
                continue

            if response.status_code not in self.retryable_status_codes or attempt >= self.max_retries:
                return response

            retry_after = response.headers.get("Retry-After")
            sleep_time = float(retry_after) if retry_after and retry_after.isdigit() else min(2 ** attempt, 30)
            time.sleep(sleep_time)

        raise RuntimeError("HTTP request failed unexpectedly.")

http_client = SentinelHubHTTPClient(auth=auth)

In [ ]:
# 6. Pydantic Request Validation Model
SUPPORTED_BANDS = {
    "B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12", "AOT", "SCL", "CLD", "SNW"
}

class MultispectralSatelliteRequest(BaseModel):
    bbox: list[float] = Field(..., description="[min_lon, min_lat, max_lon, max_lat] in WGS84")
    start_date: str = Field(..., description="YYYY-MM-DD")
    end_date: str = Field(..., description="YYYY-MM-DD")
    bands: list[str] = Field(default=["B02", "B03", "B04", "B08"], description="List of Sentinel-2 L2A band names")
    max_cloud_cover: float = Field(default=30.0, ge=0.0, le=100.0)
    width: int = Field(default=512, ge=1, le=4096)
    height: int = Field(default=512, ge=1, le=4096)
    crs: str = "EPSG:4326"

    @field_validator("bbox")
    @classmethod
    def validate_bbox(cls, value):
        if len(value) != 4: raise ValueError("bbox must contain exactly 4 values: [min_lon, min_lat, max_lon, max_lat].")
        min_lon, min_lat, max_lon, max_lat = value
        if not -180 <= min_lon <= 180 or not -180 <= max_lon <= 180: raise ValueError("Longitude must be within [-180, 180].")
        if not -90 <= min_lat <= 90 or not -90 <= max_lat <= 90: raise ValueError("Latitude must be within [-90, 90].")
        if min_lon >= max_lon or min_lat >= max_lat: raise ValueError("min coordinates must be smaller than max coordinates.")
        return value

    @field_validator("bands")
    @classmethod
    def validate_bands(cls, bands_list):
        if not bands_list: raise ValueError("bands list cannot be empty.")
        for b in bands_list:
            if b.upper() not in SUPPORTED_BANDS:
                raise ValueError(f"Unsupported band '{b}'. Valid Sentinel-2 L2A inputs are: {sorted(list(SUPPORTED_BANDS))}")
        return [b.upper() for b in bands_list]

    @field_validator("start_date", "end_date")
    @classmethod
    def validate_date_format(cls, value):
        from datetime import date
        try: date.fromisoformat(value)
        except ValueError: raise ValueError(f"Invalid date format '{value}'. Use YYYY-MM-DD.")
        return value

    @model_validator(mode="after")
    def validate_date_range(self):
        from datetime import date
        if date.fromisoformat(self.start_date) > date.fromisoformat(self.end_date):
            raise ValueError("start_date cannot be later than end_date.")
        return self

In [ ]:
# 7. STAC Catalog Search, Scene Normalization & SCL AOI Quality Inspector
def search_sentinel2_catalog(request: MultispectralSatelliteRequest, max_results: int = 20) -> list[dict]:
    datetime_range = f"{request.start_date}T00:00:00Z/{request.end_date}T23:59:59Z"
    payload = {
        "bbox": request.bbox,
        "datetime": datetime_range,
        "collections": ["sentinel-2-l2a"],
        "limit": min(10, max_results),
    }
    if request.max_cloud_cover < 100:
        payload["filter"] = f"eo:cloud_cover <= {request.max_cloud_cover}"

    all_features = []
    while len(all_features) < max_results:
        response = http_client.request("POST", CONFIG.catalog_url, json=payload)
        if not response.ok:
            try: error_data = response.json()
            except ValueError: error_data = response.text
            raise RuntimeError(f"Catalog search failed (HTTP {response.status_code}): {error_data}")

        data = response.json()
        features = data.get("features", [])
        all_features.extend(features)
        context = data.get("context", {})
        next_val = context.get("next")
        if not next_val or not features: break
        payload["next"] = next_val
        remaining = max_results - len(all_features)
        if remaining <= 0: break
        payload["limit"] = min(10, remaining)

    return all_features[:max_results]

def normalize_scene(feature: dict) -> dict:
    props = feature.get("properties", {})
    cloud_cover = props.get("eo:cloud_cover")
    return {
        "scene_id": feature.get("id"),
        "collection": "sentinel-2-l2a",
        "acquisition_time": props.get("datetime"),
        "cloud_cover": float(cloud_cover) if cloud_cover is not None else None,
        "bbox": feature.get("bbox"),
        "geometry": feature.get("geometry"),
    }

SCL_EVALSCRIPT = """//VERSION=3
function setup() {
    return { input: [{ bands: ["SCL"] }], output: { bands: 1, sampleType: "UINT8" } };
}
function evaluatePixel(sample) { return [sample.SCL]; }
"""

def evaluate_aoi_quality(bbox: list[float], acquisition_time: str) -> dict:
    acquisition_dt = datetime.fromisoformat(acquisition_time.replace("Z", "+00:00"))
    payload = {
        "input": {
            "bounds": {"bbox": bbox, "properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"}},
            "data": [{
                "type": "sentinel-2-l2a",
                "dataFilter": {"timeRange": {"from": acquisition_dt.isoformat(), "to": acquisition_dt.isoformat()}}
            }]
        },
        "output": {"width": 64, "height": 64, "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]},
        "evalscript": SCL_EVALSCRIPT,
    }
    response = http_client.request("POST", CONFIG.process_url, json=payload, content_type="application/json")
    if not response.ok:
        return {"aoi_cloud_cover": None, "aoi_cloud_shadow": None, "aoi_obstruction": None, "aoi_snow_ice": None}

    with rasterio.open(io.BytesIO(response.content)) as src:
        scl_array = src.read(1)

    total_valid = np.count_nonzero(scl_array != 0)
    if total_valid == 0:
        return {"aoi_cloud_cover": 100.0, "aoi_cloud_shadow": 0.0, "aoi_obstruction": 100.0, "aoi_snow_ice": 0.0}

    shadow_count = np.count_nonzero(scl_array == 3)
    cloud_count = np.count_nonzero(np.isin(scl_array, [8, 9, 10]))
    snow_count = np.count_nonzero(scl_array == 11)

    aoi_cloud = round(float(cloud_count / total_valid * 100.0), 2)
    aoi_shadow = round(float(shadow_count / total_valid * 100.0), 2)
    aoi_snow = round(float(snow_count / total_valid * 100.0), 2)
    aoi_obstruction = round(aoi_cloud + aoi_shadow, 2)

    return {
        "aoi_cloud_cover": aoi_cloud,
        "aoi_cloud_shadow": aoi_shadow,
        "aoi_obstruction": aoi_obstruction,
        "aoi_snow_ice": aoi_snow,
    }

def select_best_scene(scenes: list[dict], bbox: list[float], max_cloud_threshold: float = 30.0) -> dict | None:
    valid_candidates = [
        s for s in scenes if s.get("scene_id") and s.get("cloud_cover") is not None and s.get("acquisition_time")
    ]
    if not valid_candidates: return None

    valid_candidates.sort(key=lambda s: (s["cloud_cover"], s["acquisition_time"]))

    evaluated_candidates = []
    for candidate in valid_candidates[:5]:
        aoi_metrics = evaluate_aoi_quality(bbox, candidate["acquisition_time"])
        candidate_copy = dict(candidate)
        candidate_copy.update(aoi_metrics)
        evaluated_candidates.append(candidate_copy)

    def ranking_key(c):
        obs = c.get("aoi_obstruction")
        return (obs if obs is not None else 999.0, c["cloud_cover"], c["acquisition_time"])

    best = min(evaluated_candidates, key=ranking_key)
    return best

In [ ]:
# 8. Processing API Retrieval & Raster Validation
def build_multispectral_evalscript(bands: list[str]) -> str:
    band_list = ", ".join(f'"{b}"' for b in bands)
    sample_values = ", ".join(f"sample.{b}" for b in bands)
    return f"""//VERSION=3
function setup() {{
    return {{
        input: [{{
            bands: [{band_list}]
        }}],
        output: {{
            bands: {len(bands)},
            sampleType: "FLOAT32"
        }}
    }};
}}
function evaluatePixel(sample) {{
    return [{sample_values}];
}}"""

def process_multispectral_imagery(request: MultispectralSatelliteRequest, scene: dict) -> Path:
    acquisition_time = scene["acquisition_time"]
    acquisition_dt = datetime.fromisoformat(acquisition_time.replace("Z", "+00:00"))
    
    payload = {
        "input": {
            "bounds": {
                "bbox": request.bbox,
                "properties": {"crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"}
            },
            "data": [{
                "type": "sentinel-2-l2a",
                "dataFilter": {
                    "timeRange": {"from": acquisition_dt.isoformat(), "to": acquisition_dt.isoformat()},
                    "mosaickingOrder": "leastCC",
                    "maxCloudCoverage": request.max_cloud_cover,
                }
            }]
        },
        "output": {
            "width": request.width,
            "height": request.height,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]
        },
        "evalscript": build_multispectral_evalscript(request.bands),
    }

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    safe_scene_id = re.sub(r"[^A-Za-z0-9_-]", "_", scene.get("scene_id", "unknown"))
    output_path = CONFIG.output_dir / f"sentinel2_{safe_scene_id}_{request.start_date}_{request.end_date}_multispectral_{timestamp}.tif"

    response = http_client.request("POST", CONFIG.process_url, json=payload, content_type="application/json")
    if not response.ok:
        try: error_data = response.json()
        except ValueError: error_data = response.text
        raise RuntimeError(f"Processing API failed (HTTP {response.status_code}): {error_data}")

    output_path.write_bytes(response.content)
    return output_path

def validate_raster(raster_path: Path, bands: list[str]) -> dict:
    with rasterio.open(raster_path) as src:
        band_mapping = {f"Band_{i+1}": band for i, band in enumerate(bands)}
        return {
            "width": src.width,
            "height": src.height,
            "band_count": src.count,
            "dtype": str(src.dtypes[0]),
            "crs": str(src.crs),
            "band_mapping": band_mapping,
            "bounds": {"left": src.bounds.left, "bottom": src.bounds.bottom, "right": src.bounds.right, "top": src.bounds.top},
            "resolution": {"x": src.res[0], "y": src.res[1]},
            "nodata": src.nodata,
        }

In [ ]:
# 9. Core Execution Function
def fetch_multispectral_imagery(request: MultispectralSatelliteRequest) -> dict:
    raw_scenes = search_sentinel2_catalog(request, max_results=20)
    scenes = [normalize_scene(s) for s in raw_scenes]
    best_scene = select_best_scene(scenes, bbox=request.bbox, max_cloud_threshold=request.max_cloud_cover)

    if best_scene is None:
        return {
            "status": "error",
            "error": {
                "type": "no_suitable_scene",
                "message": "No suitable Sentinel-2 L2A scene was found for the requested AOI, time range and cloud threshold."
            }
        }

    raster_path = process_multispectral_imagery(request, scene=best_scene)
    raster_metadata = validate_raster(raster_path, bands=request.bands)
    
    aoi_obstruction = best_scene.get("aoi_obstruction")
    catalog_cloud = best_scene.get("cloud_cover")
    
    if aoi_obstruction is not None:
        optical_poor = aoi_obstruction > 50.0
    else:
        optical_poor = catalog_cloud is not None and catalog_cloud > 50.0

    return {
        "status": "success",
        "data": {
            "file_path": str(raster_path),
            "file_name": raster_path.name,
            "format": "GeoTIFF",
        },
        "source": {
            "provider": "Sentinel Hub",
            "collection": "sentinel-2-l2a",
            "scene_id": best_scene.get("scene_id"),
            "acquisition_time": best_scene.get("acquisition_time"),
        },
        "request": {
            "bbox": request.bbox,
            "start_date": request.start_date,
            "end_date": request.end_date,
            "modality": "multispectral",
            "bands": request.bands,
            "max_cloud_cover": request.max_cloud_cover,
        },
        "raster": raster_metadata,
        "quality": {
            "catalog_cloud_cover": catalog_cloud,
            "aoi_cloud_cover": best_scene.get("aoi_cloud_cover"),
            "aoi_cloud_shadow": best_scene.get("aoi_cloud_shadow"),
            "aoi_obstruction": aoi_obstruction,
            "aoi_snow_ice": best_scene.get("aoi_snow_ice"),
            "valid": True,
        },
        "flags": {
            "optical_quality_poor": optical_poor,
            "sar_recommended": optical_poor,
        },
    }

In [ ]:
# 10. FastMCP Tool Adapter
mcp = FastMCP("Tool 2: Multispectral Satellite Imagery Server")

@mcp.tool(
    name="fetch_multispectral_imagery",
    description="Fetch Sentinel-2 L2A calibrated surface reflectance multi-band satellite imagery for a requested bounding box and date range in GeoTIFF format with AOI-level SCL quality validation.",
)
def mcp_fetch_multispectral_imagery(
    bbox: list[float],
    start_date: str,
    end_date: str,
    bands: list[str] = ["B02", "B03", "B04", "B08"],
    max_cloud_cover: float = 30.0,
    width: int = 512,
    height: int = 512,
    crs: str = "EPSG:4326",
) -> dict:
    try:
        req = MultispectralSatelliteRequest(
            bbox=bbox,
            start_date=start_date,
            end_date=end_date,
            bands=bands,
            max_cloud_cover=max_cloud_cover,
            width=width,
            height=height,
            crs=crs,
        )
        return fetch_multispectral_imagery(req)
    except ValueError as e:
        return {"status": "error", "error": {"type": "validation_error", "message": str(e)}}
    except Exception as e:
        return {"status": "error", "error": {"type": "service_error", "message": str(e)}}